# 03 United States versus Europe (RQ2)

The regional comparison of Chapter 5 and the event-time panel of Chapter 4.

1. Random subsampling: EU-sized draws from the US sample.
2. Matched comparison: each EU event against its three nearest US events.
3. Cross-sectional regression of CAR on an EU dummy with controls.
4. Event-time panel with firm-event fixed effects, clustered by firm and calendar date (Section 4.3).

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels import PanelOLS

from src.common import (TABLES, K_VALS, ar_col, date_col, window_cols, star, p_two_sided,
                        caar_stats, clustered_ols, coef_table, load_sample)

df = load_sample()
us, eu = df[df["Region"] == "US"], df[df["Region"] == "EU"]
print(f"US {len(us)}, EU {len(eu)}")

US 545, EU 70


## Random subsampling

1,000 draws of 70 US events without replacement, and the percentile of the actual EU CAAR in the draw
distribution.

In [2]:
N_DRAWS = 1000
rng = np.random.default_rng(42)
draws = {"[-1,+1]": [], "[-5,+5]": []}
for _ in range(N_DRAWS):
    d = us.iloc[rng.choice(len(us), size=len(eu), replace=False)]
    draws["[-1,+1]"].append(caar_stats(d, -1, 1).get("EW_CAAR"))
    draws["[-5,+5]"].append(caar_stats(d, -5, 5).get("EW_CAAR"))

rows = []
for w, k1, k2 in [("[-1,+1]", -1, 1), ("[-5,+5]", -5, 5)]:
    d = np.array(draws[w], dtype=float)
    actual = caar_stats(eu, k1, k2)
    rows.append(dict(Window=w, N_draws=N_DRAWS, N_EU=len(eu), draw_mean=float(d.mean()),
                     draw_sd=float(d.std(ddof=1)), draw_p5=float(np.percentile(d, 5)),
                     draw_p95=float(np.percentile(d, 95)), EU_actual_CAAR=actual.get("EW_CAAR"),
                     EU_actual_p=actual.get("p_EW"),
                     percentile_of_actual=float((d < actual.get("EW_CAAR")).mean())))
subsampling = pd.DataFrame(rows)
subsampling.to_csv(TABLES / "subsampling.csv", index=False)
pd.DataFrame({"rep": range(N_DRAWS), "caar_1": draws["[-1,+1]"], "caar_5": draws["[-5,+5]"]}).to_csv(
    TABLES / "subsampling_draws.csv", index=False)
subsampling

,Window,N_draws,N_EU,draw_mean,draw_sd,draw_p5,draw_p95,EU_actual_CAAR,EU_actual_p,percentile_of_actual
0,"[-1,+1]",1000,70,-0.0538,0.3483,-0.6378,0.5317,0.4455,0.8833,0.907
1,"[-5,+5]",1000,70,-0.4411,0.6843,-1.5693,0.6750,0.9837,0.6586,0.983


## Matched US-EU comparison

Distance $= |\log MktCap_{EU} - \log MktCap_{US}| + 0.5\,|year_{EU} - year_{US}|$, same event type
where at least three candidates exist. Caliper at the 95th percentile of nearest-three distances.
Matching is with replacement, and the $t$-test is clustered by EU firm.

In [3]:
eu_ev, us_ev = eu.copy(), us.copy()
for x in (eu_ev, us_ev):
    x["logcap"] = np.log(x["MktCap_USDm"])
    x["year"] = x["Date"].dt.year


def candidates(e, ks):
    cand = us_ev[us_ev["Event_Type"] == e["Event_Type"]].dropna(subset=ks)
    return cand if len(cand) >= 3 else us_ev.dropna(subset=ks)


def distance(cand, e):
    return np.abs(cand["logcap"] - e["logcap"]) + 0.5 * np.abs(cand["year"] - e["year"])


ks1 = window_cols(-1, 1)
nearest = []
for _, e in eu_ev.dropna(subset=ks1).iterrows():
    cand = us_ev[us_ev["Event_Type"] == e["Event_Type"]]
    cand = cand if len(cand) >= 3 else us_ev
    nearest += distance(cand, e).nsmallest(3).tolist()
CALIPER = float(np.percentile(nearest, 95))
print(f"caliper {CALIPER:.3f}")

rows = []
for w, k1, k2 in [("[-1,+1]", -1, 1), ("[0,+3]", 0, 3), ("[-3,+3]", -3, 3), ("[-5,+5]", -5, 5)]:
    ks = window_cols(k1, k2)
    diffs, firms, n_excl = [], [], 0
    for _, e in eu_ev.dropna(subset=ks).iterrows():
        cand = candidates(e, ks)
        dist = distance(cand, e)
        within = dist[dist <= CALIPER]
        if len(within) == 0:
            n_excl += 1
            continue
        matched = cand.loc[within.nsmallest(3).index]
        diffs.append(float(e[ks].sum()) - float(matched[ks].sum(axis=1).mean()))
        firms.append(e["base"])
    d = np.array(diffs)
    n = len(d)
    frame = pd.DataFrame({"d": d, "firm": firms})
    g = frame.groupby("firm")["d"].sum() - frame.groupby("firm")["d"].count() * d.mean()
    n_clus = frame["firm"].nunique()
    se = np.sqrt(float((g ** 2).sum()) / n ** 2)
    t = d.mean() / se if se > 0 else np.nan
    p = p_two_sided(t, n_clus - 1)
    rows.append(dict(window=w, N_pairs=n, N_firms=n_clus, N_excluded_caliper=n_excl,
                     mean_diff=d.mean(), t=t, p=p, sig=star(p)))
matched = pd.DataFrame(rows)
matched.to_csv(TABLES / "matched_us_eu.csv", index=False)
matched

caliper 1.308


,window,N_pairs,N_firms,N_excluded_caliper,mean_diff,t,p,sig
0,"[-1,+1]",68,40,2,-0.0173,-0.0356,0.9718,
1,"[0,+3]",68,40,2,0.2023,0.4670,0.6431,
2,"[-3,+3]",68,40,2,0.8663,1.1295,0.2656,
3,"[-5,+5]",68,40,2,0.9749,1.1267,0.2668,


## CAR difference regression

$CAR_i = \alpha + \beta\,EU_i + \gamma\log MktCap_i + $ event-type, sector and year dummies.
Base categories are the most frequent level. Clustered by firm and date.

In [4]:
reg = df[df["MktCap_USDm"] > 0].copy()
reg["logcap"] = np.log(reg["MktCap_USDm"])
reg["year"] = reg["Date"].dt.year
dummies = []
for col, prefix in [("Event_Type", "ET"), ("Sector", "SEC"), ("year", "YR")]:
    base = reg[col].value_counts().idxmax()
    dummies.append(pd.get_dummies(reg[col], prefix=prefix, dtype=float).drop(columns=[f"{prefix}_{base}"]))
X = sm.add_constant(pd.concat([reg[["EU", "logcap"]]] + dummies, axis=1))
firm = reg["Region"].astype(str) + "_" + reg["Identifier"].astype(int).astype(str)

car_diff = pd.concat([
    coef_table(clustered_ols(reg[window_cols(k1, k2)].sum(axis=1) / 100, X, reg["Ticker"], reg["Date"],
                             firm, reg["Date"]), window=w)
    for w, k1, k2 in [("[-1,+1]", -1, 1), ("[0,+3]", 0, 3), ("[-3,+3]", -3, 3), ("[-5,+5]", -5, 5)]], ignore_index=True)
car_diff.to_csv(TABLES / "car_diff_regression.csv", index=False)
car_diff[car_diff["variable"].isin(["const", "EU", "logcap"])]

,window,variable,coef,se,t,p,sig,N
0,"[-1,+1]",const,0.0275,0.0121,2.2841,0.0227,**,615
1,"[-1,+1]",EU,0.0029,0.0058,0.4934,0.6219,,615
2,"[-1,+1]",logcap,-0.0015,0.0008,-1.8573,0.0638,*,615
21,"[0,+3]",const,0.0052,0.0161,0.3218,0.7478,,615
22,"[0,+3]",EU,0.0041,0.0059,0.6853,0.4934,,615
23,"[0,+3]",logcap,0.0002,0.0011,0.1431,0.8863,,615
42,"[-3,+3]",const,0.0400,0.0230,1.7402,0.0823,*,615
43,"[-3,+3]",EU,0.0111,0.0086,1.2898,0.1976,,615
44,"[-3,+3]",logcap,-0.0020,0.0015,-1.3487,0.1779,,615
63,"[-5,+5]",const,0.0528,0.0245,2.1577,0.0314,**,615


## Event-time panel

Long panel of $AR_{it}$, $t=-5,\dots,+5$, with event-day indicators $D_{t=m}$ and interactions
$D_{t=m} \times EU_i$. Each firm-event has its own fixed effect $\mu_i$. Standard errors are clustered by firm
and calendar date.

- Baseline (Equation 4.10): all $D_{t=m}$ ($m \neq -1$) plus $D_{t=0} \times EU_i$.
- Fully interacted (Equation 4.11): all $D_{t=m}$ and $D_{t=m} \times EU_i$, reference day $-1$.
- Fully interacted with reference day $-5$: day $-1$ has the highest mean abnormal return, so the day $-1$
  reference shifts every coefficient down.

In [5]:
long = pd.DataFrame([dict(event_id=r["event_id"], k=k, date_k=pd.Timestamp(r[date_col(k)]),
                          AR=r[ar_col(k)] / 100, EU=r["EU"], firm=f"{r['Region']}_{int(r['Identifier'])}")
                     for _, r in df.iterrows() for k in K_VALS if pd.notna(r[ar_col(k)])])


def event_panel(ref, prefix="d"):
    p = long.copy()
    ks = [k for k in K_VALS if k != ref]
    for k in K_VALS:
        p[f"{prefix}_k{k:+d}"] = (p["k"] == k).astype(float)
    for k in ks:
        p[f"{prefix}_k{k:+d}_EU"] = p[f"{prefix}_k{k:+d}"] * p["EU"]
    p = p.set_index(["event_id", "date_k"])
    clusters = pd.DataFrame({"firm": p["firm"].to_numpy(), "date": p.index.get_level_values("date_k")},
                            index=p.index)
    return p, ks, clusters


def fit(p, cols, clusters):
    return PanelOLS(p["AR"], p[cols], entity_effects=True, drop_absorbed=True).fit(
        cov_type="clustered", clusters=clusters)


panel, ks, clusters = event_panel(-1)
static = coef_table(fit(panel, [f"d_k{k:+d}" for k in ks] + ["d_k+0_EU"], clusters))
dynamic = coef_table(fit(panel, [f"d_k{k:+d}" for k in ks] + [f"d_k{k:+d}_EU" for k in ks], clusters))
panel5, ks5, clusters5 = event_panel(-5, prefix="r5")
dynamic5 = coef_table(fit(panel5, [f"r5_k{k:+d}" for k in ks5] + [f"r5_k{k:+d}_EU" for k in ks5], clusters5))
dynamic5["variable"] = dynamic5["variable"].str.replace("r5_", "d_")

static.to_csv(TABLES / "panel_static.csv", index=False)
dynamic.to_csv(TABLES / "panel_dynamic.csv", index=False)
dynamic5.to_csv(TABLES / "panel_dynamic_ref5.csv", index=False)
static[static["variable"].isin(["d_k+0", "d_k+0_EU"])]

,variable,coef,se,t,p,sig,N
4,d_k+0,-0.0018,0.0012,-1.4845,0.1377,,6765
10,d_k+0_EU,-0.0011,0.0024,-0.4627,0.6436,,6765


In [6]:
dynamic5

,variable,coef,se,t,p,sig,N
0,d_k-4,-2.5569e-04,0.0012,-0.2213,0.8249,,6765
1,d_k-3,-1.5225e-03,0.0012,-1.2324,0.2178,,6765
2,d_k-2,-4.2251e-04,0.0010,-0.4141,0.6788,,6765
3,d_k-1,1.6437e-03,0.0012,1.3512,0.1767,,6765
4,d_k+0,-2.8895e-04,0.0015,-0.1992,0.8421,,6765
5,d_k+1,-1.3288e-03,0.0011,-1.1589,0.2466,,6765
6,d_k+2,-7.1545e-05,0.0010,-0.0699,0.9442,,6765
7,d_k+3,-5.9178e-04,0.0012,-0.5133,0.6078,,6765
8,d_k+4,-2.6043e-04,0.0014,-0.1835,0.8544,,6765
9,d_k+5,1.2519e-03,0.0014,0.8692,0.3848,,6765
